# BiLSTM Persian POS Tagging

**FarsiSyntax Lab**

Research notebook for the BiLSTM baseline and Persian POS tagging experiments.

## Research context

The project compares neural POS tagging on formal and informal Persian.

**Important:** The original notebook was executed in its research environment. Some code cells reference Google Colab paths or external datasets. Adjust these paths and install dependencies before rerunning the experiment.

Execution outputs have been removed from this publication copy. Original training logs are retained locally. See the repository README and reports for recorded evaluation results.


In [ ]:

from pathlib import Path
from collections import Counter
from conllu import parse_incr
import pandas as pd
import torch
import transformers

# ==========================================
# FARSI SYNTAX LAB - EXPERIMENT 01
# PERSIAN LINGUISTIC DATA INSPECTION
# ==========================================

ROOT = Path.cwd()

if not (ROOT / "data/raw/formal/train.conllu").exists():
    ROOT = ROOT.parent

assert (ROOT / "data/raw/formal/train.conllu").exists()

print("=== ENVIRONMENT ===")
print("Project:", ROOT)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

datasets = {}
records = []
pos_inventories = {}

for register in ["formal", "informal"]:
    for split in ["train", "dev", "test"]:

        path = ROOT / "data/raw" / register / f"{split}.conllu"
        assert path.exists(), f"Missing: {path}"

        sentence_count = 0
        token_count = 0
        pos_counts = Counter()
        zwnj_count = 0
        example = None

        with path.open("r", encoding="utf-8-sig") as f:
            for sentence in parse_incr(f):

                # Exclude multiword range entries and empty nodes
                words = [
                    token for token in sentence
                    if isinstance(token["id"], int)
                ]

                if not words:
                    continue

                sentence_count += 1
                token_count += len(words)

                for word in words:
                    pos_counts[word["upos"]] += 1
                    if "\u200c" in word["form"]:
                        zwnj_count += 1

                if example is None:
                    example = [
                        {
                            "Word": word["form"],
                            "POS": word["upos"],
                            "Dependency": word["deprel"]
                        }
                        for word in words[:12]
                    ]

        datasets[(register, split)] = path
        pos_inventories[(register, split)] = set(pos_counts)

        records.append({
            "Register": register,
            "Split": split,
            "Sentences": sentence_count,
            "Words": token_count,
            "POS tags": len(pos_counts),
            "ZWNJ words": zwnj_count
        })

        print(
            f"[OK] {register:8s} {split:5s} | "
            f"{sentence_count:6,d} sentences | "
            f"{token_count:7,d} words"
        )

        if split == "train":
            print(f"\n=== {register.upper()} SAMPLE ===")
            display(pd.DataFrame(example))
            print()

summary_df = pd.DataFrame(records)

print("\n=== COMPLETE DATASET SUMMARY ===")
display(summary_df)

formal_tags = pos_inventories[("formal", "train")]

print("\n=== FORMAL TRAIN POS LABELS ===")
print(sorted(formal_tags))

for register in ["formal", "informal"]:
    for split in ["dev", "test"]:
        unseen = pos_inventories[(register, split)] - formal_tags
        print(
            f"Unseen POS tags in {register} {split}:",
            sorted(unseen)
        )

print("\nEXPERIMENT 01 - DATA INSPECTION COMPLETE")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 01
# CELL 2: BiLSTM DATA PREPARATION
# ============================================

from collections import Counter
from conllu import parse_incr
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import random
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# CPU optimization
torch.set_num_threads(min(4, torch.get_num_threads()))

# --------------------------------------------
# 1. Load annotated Persian sentences
# --------------------------------------------

def load_sentences(path):
    sentences = []

    with open(path, "r", encoding="utf-8-sig") as file:
        for sentence in parse_incr(file):

            words = [
                (token["form"], token["upos"])
                for token in sentence
                if isinstance(token["id"], int)
            ]

            if words:
                sentences.append(words)

    return sentences


formal_train = load_sentences(
    ROOT / "data/raw/formal/train.conllu"
)

formal_dev = load_sentences(
    ROOT / "data/raw/formal/dev.conllu"
)

informal_dev = load_sentences(
    ROOT / "data/raw/informal/dev.conllu"
)

print("=== DATA LOADED ===")
print("Formal train:", len(formal_train))
print("Formal dev:", len(formal_dev))
print("Informal dev:", len(informal_dev))


# --------------------------------------------
# 2. Build vocabulary from training only
# --------------------------------------------

word_counts = Counter(
    word
    for sentence in formal_train
    for word, pos in sentence
)

MIN_FREQ = 2

word2id = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, frequency in word_counts.items():
    if frequency >= MIN_FREQ:
        word2id[word] = len(word2id)

id2word = {
    index: word
    for word, index in word2id.items()
}

# POS categories learned from training only
pos_tags = sorted({
    pos
    for sentence in formal_train
    for word, pos in sentence
})

pos2id = {
    pos: index
    for index, pos in enumerate(pos_tags)
}

id2pos = {
    index: pos
    for pos, index in pos2id.items()
}

PAD_ID = 0
UNK_ID = 1
IGNORE_INDEX = -100

print("\n=== VOCABULARY ===")
print("Unique training words:", len(word_counts))
print("Vocabulary size:", len(word2id))
print("POS categories:", len(pos2id))
print("POS labels:", pos_tags)


# --------------------------------------------
# 3. Convert sentences to tensors
# --------------------------------------------

class PersianPOSDataset(Dataset):

    def __init__(self, sentences):
        self.sentences = sentences

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, index):
        sentence = self.sentences[index]

        word_ids = [
            word2id.get(word, UNK_ID)
            for word, pos in sentence
        ]

        label_ids = [
            pos2id[pos]
            for word, pos in sentence
        ]

        return (
            torch.tensor(word_ids, dtype=torch.long),
            torch.tensor(label_ids, dtype=torch.long)
        )


def collate_batch(batch):

    words, labels = zip(*batch)

    lengths = torch.tensor(
        [len(sentence) for sentence in words],
        dtype=torch.long
    )

    padded_words = pad_sequence(
        words,
        batch_first=True,
        padding_value=PAD_ID
    )

    padded_labels = pad_sequence(
        labels,
        batch_first=True,
        padding_value=IGNORE_INDEX
    )

    return padded_words, padded_labels, lengths


# --------------------------------------------
# 4. Create DataLoaders
# --------------------------------------------

BATCH_SIZE = 32

train_dataset = PersianPOSDataset(formal_train)
formal_dev_dataset = PersianPOSDataset(formal_dev)
informal_dev_dataset = PersianPOSDataset(informal_dev)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=0
)

formal_dev_loader = DataLoader(
    formal_dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0
)

informal_dev_loader = DataLoader(
    informal_dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0
)


# --------------------------------------------
# 5. Measure out-of-vocabulary words
# --------------------------------------------

def calculate_oov(sentences):
    total = 0
    unknown = 0

    for sentence in sentences:
        for word, pos in sentence:
            total += 1

            if word not in word2id:
                unknown += 1

    return unknown / total if total else 0


print("\n=== VOCABULARY COVERAGE ===")
print(
    f"Formal dev OOV: "
    f"{calculate_oov(formal_dev):.2%}"
)
print(
    f"Informal dev OOV: "
    f"{calculate_oov(informal_dev):.2%}"
)


# --------------------------------------------
# 6. Verify one training batch
# --------------------------------------------

batch_words, batch_labels, batch_lengths = next(
    iter(train_loader)
)

assert batch_words.shape == batch_labels.shape
assert batch_words.shape[0] <= BATCH_SIZE
assert len(pos2id) == 16

print("\n=== TRAINING BATCH VERIFICATION ===")
print("Word tensor:", tuple(batch_words.shape))
print("Label tensor:", tuple(batch_labels.shape))
print("Sentence lengths:", tuple(batch_lengths.shape))

print("\n=== TRAINING DATA READY ===")
print("Device:", DEVICE)
print("Training batches:", len(train_loader))
print("Formal validation batches:", len(formal_dev_loader))
print("Informal validation batches:", len(informal_dev_loader))
print("Test datasets: NOT LOADED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 01
# CELL 3: BIDIRECTIONAL LSTM MODEL
# ============================================

import torch
import torch.nn as nn
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)

# --------------------------------------------
# 1. Model configuration
# --------------------------------------------

VOCAB_SIZE = len(word2id)
NUM_TAGS = len(pos2id)

EMBEDDING_DIM = 128
HIDDEN_DIM = 128
DROPOUT = 0.3


# --------------------------------------------
# 2. Define the BiLSTM network
# --------------------------------------------

class PersianBiLSTMTagger(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_tags,
        pad_id,
        dropout=0.3
    ):
        super().__init__()

        # Convert word IDs into learned vectors
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_id
        )

        self.dropout = nn.Dropout(dropout)

        # Read Persian sentences in both directions
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )

        # Predict one grammatical category per word
        self.classifier = nn.Linear(
            hidden_dim * 2,
            num_tags
        )

    def forward(self, word_ids, lengths):

        embedded = self.embedding(word_ids)
        embedded = self.dropout(embedded)

        # Skip padding during recurrent computation
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=word_ids.size(1)
        )

        output = self.dropout(output)

        logits = self.classifier(output)

        return logits


# --------------------------------------------
# 3. Initialize neural network
# --------------------------------------------

model = PersianBiLSTMTagger(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_tags=NUM_TAGS,
    pad_id=PAD_ID,
    dropout=DROPOUT
).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    ignore_index=IGNORE_INDEX
)

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("=== BILSTM ARCHITECTURE ===")
print(model)

print("\n=== MODEL PARAMETERS ===")
print("Vocabulary size:", VOCAB_SIZE)
print("Embedding dimension:", EMBEDDING_DIM)
print("Hidden dimension:", HIDDEN_DIM)
print("Output categories:", NUM_TAGS)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Device:", DEVICE)


# --------------------------------------------
# 4. Verify the forward pass
# --------------------------------------------

model.train()

sample_words = batch_words.to(DEVICE)
sample_labels = batch_labels.to(DEVICE)

logits = model(
    sample_words,
    batch_lengths
)

assert logits.shape == (
    sample_words.shape[0],
    sample_words.shape[1],
    NUM_TAGS
)

print("\n=== FORWARD PASS ===")
print("Input shape:", tuple(sample_words.shape))
print("Output shape:", tuple(logits.shape))


# --------------------------------------------
# 5. Verify loss and backward pass
# --------------------------------------------

loss = criterion(
    logits.reshape(-1, NUM_TAGS),
    sample_labels.reshape(-1)
)

assert torch.isfinite(loss).item()

loss.backward()

assert model.embedding.weight.grad is not None
assert torch.isfinite(
    model.embedding.weight.grad
).all().item()

print("\n=== TRAINING READINESS ===")
print("Initial batch loss:", round(loss.item(), 4))
print("Backward propagation:", "SUCCESS")

# Clear verification gradients before training
model.zero_grad(set_to_none=True)

print("\nBILSTM MODEL VERIFICATION COMPLETE")
print("Training epochs executed: 0")
print("Test datasets: NOT USED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 01
# CELL 4: TRAINING THE PERSIAN BILSTM
# ============================================

import time
import copy
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

# --------------------------------------------
# 1. Training configuration
# --------------------------------------------

LEARNING_RATE = 0.001
MAX_EPOCHS = 10
PATIENCE = 3

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.0001
)

history = []
best_f1 = -1.0
best_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

MODEL_PATH = ROOT / "models/bilstm_formal_best.pt"
HISTORY_PATH = ROOT / "reports/bilstm_training_history.csv"

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)


# --------------------------------------------
# 2. Train for one epoch
# --------------------------------------------

def train_one_epoch(model, loader):

    model.train()

    total_loss = 0.0
    total_tokens = 0
    correct_tokens = 0

    progress = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for words, labels, lengths in progress:

        words = words.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = model(words, lengths)

        loss = criterion(
            logits.reshape(-1, NUM_TAGS),
            labels.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        valid_mask = labels != IGNORE_INDEX
        count = valid_mask.sum().item()

        total_loss += loss.item() * count
        total_tokens += count

        predictions = logits.argmax(dim=-1)

        correct_tokens += (
            (predictions == labels) & valid_mask
        ).sum().item()

    return (
        total_loss / total_tokens,
        correct_tokens / total_tokens
    )


# --------------------------------------------
# 3. Evaluate token-level predictions
# --------------------------------------------

@torch.inference_mode()
def evaluate_model(model, loader):

    model.eval()

    total_loss = 0.0
    total_tokens = 0

    true_labels = []
    predicted_labels = []

    for words, labels, lengths in loader:

        words = words.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(words, lengths)

        loss = criterion(
            logits.reshape(-1, NUM_TAGS),
            labels.reshape(-1)
        )

        predictions = logits.argmax(dim=-1)

        valid_mask = labels != IGNORE_INDEX
        count = valid_mask.sum().item()

        total_loss += loss.item() * count
        total_tokens += count

        true_labels.extend(
            labels[valid_mask].cpu().tolist()
        )

        predicted_labels.extend(
            predictions[valid_mask].cpu().tolist()
        )

    accuracy = accuracy_score(
        true_labels,
        predicted_labels
    )

    macro_f1 = f1_score(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_TAGS)),
        average="macro",
        zero_division=0
    )

    return {
        "loss": total_loss / total_tokens,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "tokens": total_tokens
    }


# --------------------------------------------
# 4. Train with early stopping
# --------------------------------------------

print("=== PERSIAN BILSTM TRAINING ===")
print("Training sentences:", len(formal_train))
print("Validation sentences:", len(formal_dev))
print("Vocabulary:", VOCAB_SIZE)
print("POS categories:", NUM_TAGS)
print("Device:", DEVICE)
print("Maximum epochs:", MAX_EPOCHS)

for epoch in range(1, MAX_EPOCHS + 1):

    start_time = time.perf_counter()

    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader
    )

    # Model selection uses FORMAL DEV ONLY
    validation = evaluate_model(
        model,
        formal_dev_loader
    )

    elapsed = time.perf_counter() - start_time

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "formal_val_loss": validation["loss"],
        "formal_val_accuracy": validation["accuracy"],
        "formal_val_macro_f1": validation["macro_f1"],
        "seconds": elapsed
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {validation['loss']:.4f} | "
        f"Val Acc: {validation['accuracy']:.4f} | "
        f"Val Macro F1: {validation['macro_f1']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

    improved = (
        validation["macro_f1"] > best_f1 + 1e-6
        or (
            abs(validation["macro_f1"] - best_f1) <= 1e-6
            and validation["loss"] < best_loss
        )
    )

    if improved:

        best_f1 = validation["macro_f1"]
        best_loss = validation["loss"]
        best_epoch = epoch

        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "word2id": word2id,
                "pos2id": pos2id,
                "embedding_dim": EMBEDDING_DIM,
                "hidden_dim": HIDDEN_DIM,
                "dropout": DROPOUT,
                "best_epoch": best_epoch,
                "validation_macro_f1": best_f1,
                "seed": SEED
            },
            MODEL_PATH
        )

        print("  [SAVED] New best model", flush=True)

    else:
        epochs_without_improvement += 1

    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False
    )

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping activated.")
        break


# --------------------------------------------
# 5. Restore the best checkpoint
# --------------------------------------------

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

best_validation = evaluate_model(
    model,
    formal_dev_loader
)

print("\n=== FINAL FORMAL VALIDATION RESULTS ===")
print("Best epoch:", best_epoch)
print(
    f"Token accuracy: "
    f"{best_validation['accuracy']:.4f}"
)
print(
    f"Macro F1: "
    f"{best_validation['macro_f1']:.4f}"
)
print("Validation tokens:", best_validation["tokens"])
print("Checkpoint:", MODEL_PATH)
print("Training history:", HISTORY_PATH)

print("\nEXPERIMENT 01 - BILSTM TRAINING COMPLETE")
print("Informal validation: NOT USED FOR MODEL SELECTION")
print("Test datasets: NOT USED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 01
# CELL 5: CROSS-REGISTER EVALUATION
# ============================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# --------------------------------------------
# 1. Restore best trained checkpoint
# --------------------------------------------

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("=== SELECTED BILSTM MODEL ===")
print("Best epoch:", checkpoint["best_epoch"])
print(
    "Formal validation Macro F1:",
    round(checkpoint["validation_macro_f1"], 4)
)


# --------------------------------------------
# 2. Collect predictions and OOV indicators
# --------------------------------------------

@torch.inference_mode()
def evaluate_register(loader):

    model.eval()

    all_true = []
    all_pred = []
    all_oov = []

    for words, labels, lengths in loader:

        logits = model(
            words.to(DEVICE),
            lengths
        )

        predictions = logits.argmax(-1).cpu()

        valid = labels != IGNORE_INDEX

        all_true.extend(
            labels[valid].tolist()
        )

        all_pred.extend(
            predictions[valid].tolist()
        )

        all_oov.extend(
            (words[valid] == UNK_ID).tolist()
        )

    y_true = np.asarray(all_true)
    y_pred = np.asarray(all_pred)
    is_oov = np.asarray(all_oov, dtype=bool)

    assert len(y_true) == len(y_pred) == len(is_oov)

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_TAGS)),
        target_names=pos_tags,
        output_dict=True,
        zero_division=0
    )

    return {
        "tokens": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=list(range(NUM_TAGS)),
            average="macro",
            zero_division=0
        ),
        "oov_rate": is_oov.mean(),
        "known_accuracy": accuracy_score(
            y_true[~is_oov],
            y_pred[~is_oov]
        ),
        "oov_accuracy": (
            accuracy_score(
                y_true[is_oov],
                y_pred[is_oov]
            )
            if is_oov.any() else float("nan")
        ),
        "report": report
    }


# --------------------------------------------
# 3. Evaluate both registers
# --------------------------------------------

formal_results = evaluate_register(
    formal_dev_loader
)

informal_results = evaluate_register(
    informal_dev_loader
)

summary = pd.DataFrame([
    {
        "Register": "Formal",
        **{
            k: v for k, v in formal_results.items()
            if k != "report"
        }
    },
    {
        "Register": "Informal",
        **{
            k: v for k, v in informal_results.items()
            if k != "report"
        }
    }
])

print("\n=== FORMAL VS INFORMAL RESULTS ===")
print(
    summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)


# --------------------------------------------
# 4. Measure cross-register performance gap
# --------------------------------------------

accuracy_gap = (
    formal_results["accuracy"]
    - informal_results["accuracy"]
)

macro_f1_gap = (
    formal_results["macro_f1"]
    - informal_results["macro_f1"]
)

print("\n=== CROSS-REGISTER GAP ===")
print(
    f"Accuracy gap: {accuracy_gap * 100:.2f} "
    "percentage points"
)
print(
    f"Macro F1 gap: {macro_f1_gap:.4f}"
)


# --------------------------------------------
# 5. Compare individual POS categories
# --------------------------------------------

pos_comparison = []

for tag in pos_tags:

    formal_tag = formal_results["report"][tag]
    informal_tag = informal_results["report"][tag]

    pos_comparison.append({
        "POS": tag,
        "Formal F1": formal_tag["f1-score"],
        "Informal F1": informal_tag["f1-score"],
        "F1 Change": (
            informal_tag["f1-score"]
            - formal_tag["f1-score"]
        ),
        "Formal Support": int(formal_tag["support"]),
        "Informal Support": int(informal_tag["support"])
    })

pos_df = pd.DataFrame(pos_comparison)

print("\n=== POS CATEGORY COMPARISON ===")
print(
    pos_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)


# --------------------------------------------
# 6. Save research results
# --------------------------------------------

report_dir = ROOT / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

summary_path = (
    report_dir /
    "bilstm_register_comparison.csv"
)

pos_path = (
    report_dir /
    "bilstm_pos_category_comparison.csv"
)

summary.to_csv(summary_path, index=False)
pos_df.to_csv(pos_path, index=False)

print("\n=== EXPERIMENT 01 RESULTS SAVED ===")
print("Summary:", summary_path)
print("POS comparison:", pos_path)
print("Checkpoint:", MODEL_PATH)

print("\nCROSS-REGISTER EVALUATION COMPLETE")
print("Informal data used for evaluation only.")
print("Test datasets: NOT USED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 01
# CELL 6: LINGUISTIC ERROR ANALYSIS
# ============================================

from collections import Counter
import pandas as pd
import torch

model.eval()

@torch.inference_mode()
def collect_predictions(loader, sentences):

    rows = []
    sentence_index = 0

    for words, labels, lengths in loader:

        logits = model(
            words.to(DEVICE),
            lengths
        )

        predictions = logits.argmax(-1).cpu()

        for batch_index, length in enumerate(lengths.tolist()):

            sentence = sentences[sentence_index]

            assert len(sentence) == length

            text = " ".join(
                word for word, pos in sentence
            )

            for token_index, (word, gold) in enumerate(sentence):

                predicted = id2pos[
                    predictions[batch_index, token_index].item()
                ]

                assert (
                    labels[batch_index, token_index].item()
                    == pos2id[gold]
                )

                rows.append({
                    "word": word,
                    "gold": gold,
                    "predicted": predicted,
                    "correct": gold == predicted,
                    "oov": word2id.get(word, UNK_ID) == UNK_ID,
                    "sentence": text
                })

            sentence_index += 1

    assert sentence_index == len(sentences)

    return pd.DataFrame(rows)


errors_df = collect_predictions(
    informal_dev_loader,
    informal_dev
)

print("=== INFORMAL PERSIAN ERROR ANALYSIS ===")
print("Total tokens:", len(errors_df))
print("Incorrect predictions:", (~errors_df["correct"]).sum())
print("Correct predictions:", errors_df["correct"].sum())


# --------------------------------------------
# 1. Most frequent grammatical confusions
# --------------------------------------------

confusions = (
    errors_df[~errors_df["correct"]]
    .groupby(["gold", "predicted"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("\n=== TOP 12 POS CONFUSIONS ===")
print(confusions.head(12).to_string(index=False))


# --------------------------------------------
# 2. Inspect challenging grammatical categories
# --------------------------------------------

TARGET_TAGS = ["PART", "ADV", "AUX", "PROPN", "X"]

for tag in TARGET_TAGS:

    formal_words = [
        word
        for sentence in formal_train
        for word, pos in sentence
        if pos == tag
    ]

    informal_words = errors_df[
        errors_df["gold"] == tag
    ]

    mistakes = informal_words[
        ~informal_words["correct"]
    ]

    print(f"\n=== {tag}: LINGUISTIC INSPECTION ===")

    print("Formal train tokens:", len(formal_words))
    print("Informal dev tokens:", len(informal_words))

    print(
        "Common formal words:",
        Counter(formal_words).most_common(8)
    )

    print(
        "Common informal words:",
        Counter(informal_words["word"]).most_common(8)
    )

    if len(mistakes):
        print(
            "Most common incorrect tags:",
            mistakes["predicted"]
            .value_counts()
            .head(4)
            .to_dict()
        )

        print("\nExample mistakes:")

        for _, row in mistakes.head(3).iterrows():

            print(
                f"Word: {row['word']} | "
                f"Gold: {row['gold']} | "
                f"Predicted: {row['predicted']} | "
                f"OOV: {row['oov']}"
            )

            print("Context:", row["sentence"][:160])
            print()


# --------------------------------------------
# 3. Save aggregated error statistics
# --------------------------------------------

output = (
    ROOT / "reports/bilstm_informal_confusions.csv"
)

confusions.to_csv(output, index=False)

print("\n=== ANALYSIS COMPLETE ===")
print("Confusion report:", output)
print("Model checkpoint: UNCHANGED")
print("Test datasets: NOT USED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 02
# CELL 7: PARSBERT TOKENIZATION & ALIGNMENT
# ============================================

from transformers import AutoTokenizer

MODEL_NAME = "HooshvareLab/bert-fa-zwnj-base"
IGNORE_INDEX = -100

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

assert tokenizer.is_fast, "Fast tokenizer required."

print("=== PARSBERT TOKENIZER ===")
print("Model:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print("Fast tokenizer:", tokenizer.is_fast)
print("Maximum length:", tokenizer.model_max_length)


# Assign each word's POS label to its first subword.
# Subsequent subwords and special tokens are ignored.

def align_pos_labels(sentence):
    words = [word for word, tag in sentence]
    tags = [tag for word, tag in sentence]

    encoding = tokenizer(
        words,
        is_split_into_words=True,
        truncation=False,
        add_special_tokens=True
    )

    word_ids = encoding.word_ids()
    aligned = []
    previous_word_id = None

    for word_id in word_ids:

        if word_id is None:
            aligned.append(IGNORE_INDEX)

        elif word_id != previous_word_id:
            aligned.append(pos2id[tags[word_id]])

        else:
            aligned.append(IGNORE_INDEX)

        previous_word_id = word_id

    assert len(encoding["input_ids"]) == len(aligned)

    covered_words = {
        word_id
        for word_id in word_ids
        if word_id is not None
    }

    return encoding, aligned, covered_words


# Select actual examples containing Persian half-spaces

def find_example(sentences):
    for sentence in sentences:
        words = [word for word, tag in sentence]

        if (
            5 <= len(words) <= 35
            and any("\u200c" in word for word in words)
        ):
            return sentence

    return sentences[0]


examples = {
    "FORMAL": find_example(formal_train),
    "INFORMAL": find_example(informal_dev)
}

for register, sentence in examples.items():

    encoding, aligned, covered = align_pos_labels(sentence)

    words = [word for word, tag in sentence]
    word_ids = encoding.word_ids()

    print(f"\n=== {register} PERSIAN ===")
    print("Sentence:", " ".join(words))
    print("Original words:", len(words))
    print("Subword tokens:", len(encoding["input_ids"]))
    print("Words aligned:", len(covered))

    print("\nFirst 25 subword annotations:")

    pieces = tokenizer.convert_ids_to_tokens(
        encoding["input_ids"]
    )

    for piece, word_id, label in list(
        zip(pieces, word_ids, aligned)
    )[:25]:

        tag = (
            id2pos[label]
            if label != IGNORE_INDEX
            else "IGNORED"
        )

        print(
            f"{piece:20s} | "
            f"word_id={str(word_id):4s} | "
            f"POS={tag}"
        )


# Audit a representative sample without touching test sets

print("\n=== TOKENIZATION COVERAGE AUDIT ===")

for register, sentences in [
    ("Formal train", formal_train[:200]),
    ("Formal dev", formal_dev[:200]),
    ("Informal dev", informal_dev[:200])
]:

    total_words = 0
    aligned_words = 0
    above_256 = 0

    for sentence in sentences:
        encoding, labels, covered = align_pos_labels(
            sentence
        )

        total_words += len(sentence)
        aligned_words += len(covered)

        if len(encoding["input_ids"]) > 256:
            above_256 += 1

    print(f"\n{register}")
    print("Sentences checked:", len(sentences))
    print("Original words:", total_words)
    print("Aligned words:", aligned_words)
    print("Sentences over 256 tokens:", above_256)

print("\nPARSBERT TOKENIZATION VERIFICATION COMPLETE")
print("Model weights: NOT DOWNLOADED")
print("Test datasets: NOT USED")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 02
# CELL 8: FULL PARSBERT LENGTH AUDIT
# ============================================

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

corpora = {
    "Formal train": formal_train,
    "Formal dev": formal_dev,
    "Informal dev": informal_dev
}

BATCH_SIZE_AUDIT = 256
MAX_LENGTH = 256

records = []

print("=== PARSBERT FULL CORPUS AUDIT ===")

for name, sentences in corpora.items():

    lengths = []
    missing_alignment = 0

    for start in tqdm(
        range(0, len(sentences), BATCH_SIZE_AUDIT),
        desc=name
    ):

        batch = sentences[
            start:start + BATCH_SIZE_AUDIT
        ]

        words_batch = [
            [word for word, tag in sentence]
            for sentence in batch
        ]

        encoded = tokenizer(
            words_batch,
            is_split_into_words=True,
            truncation=False,
            padding=False,
            add_special_tokens=True,
            return_length=True
        )

        lengths.extend(encoded["length"])

        for i, words in enumerate(words_batch):

            covered = {
                index
                for index in encoded.word_ids(batch_index=i)
                if index is not None
            }

            if len(covered) != len(words):
                missing_alignment += 1

    lengths = np.asarray(lengths)

    record = {
        "Corpus": name,
        "Sentences": len(sentences),
        "Mean tokens": round(lengths.mean(), 1),
        "P95 tokens": int(np.percentile(lengths, 95)),
        "Max tokens": int(lengths.max()),
        "Over 256": int((lengths > 256).sum()),
        "Over 512": int((lengths > 512).sum()),
        "Alignment failures": missing_alignment
    }

    records.append(record)

    print(f"\n{name}")
    print("Sentences:", len(sentences))
    print("Average length:", record["Mean tokens"])
    print("Maximum length:", record["Max tokens"])
    print("Over 256:", record["Over 256"])
    print("Over 512:", record["Over 512"])
    print("Alignment failures:", missing_alignment)

summary = pd.DataFrame(records)

print("\n=== COMPLETE PARSBERT LENGTH REPORT ===")
print(summary.to_string(index=False))

output = ROOT / "reports/parsbert_length_audit.csv"
summary.to_csv(output, index=False)

assert summary["Alignment failures"].sum() == 0, (
    "Token alignment requires investigation."
)

print("\nFULL TOKENIZATION AUDIT COMPLETE")
print("Saved:", output)
print("Model training: NOT STARTED")
print("Test datasets: NOT USED")


In [ ]:

# =============================================
# FARSI SYNTAX LAB - EXPERIMENT 02
# CELL 9: PARSBERT TRAINING DATA PIPELINE
# =============================================

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorForTokenClassification
from tqdm.auto import tqdm

MAX_LENGTH = 256
IGNORE_INDEX = -100

# ---------------------------------------------
# 1. Tokenize sentences and align POS labels
# ---------------------------------------------

class ParsBERTPOSDataset(Dataset):

    def __init__(self, sentences, tokenizer, name):
        self.examples = []
        self.name = name

        batch_size = 256

        for start in tqdm(
            range(0, len(sentences), batch_size),
            desc=f"Preparing {name}"
        ):
            batch = sentences[start:start + batch_size]

            words_batch = [
                [word for word, pos in sentence]
                for sentence in batch
            ]

            tags_batch = [
                [pos for word, pos in sentence]
                for sentence in batch
            ]

            encodings = tokenizer(
                words_batch,
                is_split_into_words=True,
                truncation=True,
                max_length=MAX_LENGTH,
                padding=False
            )

            for i, tags in enumerate(tags_batch):

                word_ids = encodings.word_ids(
                    batch_index=i
                )

                labels = []
                previous_word_id = None

                for word_id in word_ids:

                    if word_id is None:
                        labels.append(IGNORE_INDEX)

                    elif word_id != previous_word_id:
                        labels.append(
                            pos2id[tags[word_id]]
                        )

                    else:
                        labels.append(IGNORE_INDEX)

                    previous_word_id = word_id

                # Every original word must contribute
                # exactly one supervised POS label.
                supervised = sum(
                    label != IGNORE_INDEX
                    for label in labels
                )

                assert supervised == len(tags), (
                    f"Lost word alignment in {name}"
                )

                example = {
                    "input_ids": encodings["input_ids"][i],
                    "attention_mask": encodings[
                        "attention_mask"
                    ][i],
                    "labels": labels
                }

                self.examples.append(example)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


# ---------------------------------------------
# 2. Build the three training/evaluation sets
# ---------------------------------------------

bert_train_dataset = ParsBERTPOSDataset(
    formal_train,
    tokenizer,
    "Formal train"
)

bert_formal_dev_dataset = ParsBERTPOSDataset(
    formal_dev,
    tokenizer,
    "Formal dev"
)

bert_informal_dev_dataset = ParsBERTPOSDataset(
    informal_dev,
    tokenizer,
    "Informal dev"
)


# ---------------------------------------------
# 3. Dynamic padding
# ---------------------------------------------

bert_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=IGNORE_INDEX,
    return_tensors="pt"
)

bert_train_loader = DataLoader(
    bert_train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=bert_collator,
    num_workers=0
)


# ---------------------------------------------
# 4. Verify a batch
# ---------------------------------------------

batch = next(iter(bert_train_loader))

print("\n=== PARSBERT TRAINING BATCH ===")
print("Input IDs:", tuple(batch["input_ids"].shape))
print(
    "Attention mask:",
    tuple(batch["attention_mask"].shape)
)
print("POS labels:", tuple(batch["labels"].shape))

assert (
    batch["input_ids"].shape
    == batch["labels"].shape
    == batch["attention_mask"].shape
)

assert batch["input_ids"].shape[1] <= MAX_LENGTH

valid_labels = batch["labels"] != IGNORE_INDEX

print(
    "Supervised words in batch:",
    valid_labels.sum().item()
)

print(
    "Padding tokens:",
    (batch["attention_mask"] == 0).sum().item()
)


# ---------------------------------------------
# 5. Verify all datasets
# ---------------------------------------------

print("\n=== PARSBERT DATASET VERIFICATION ===")

for name, dataset in [
    ("Formal train", bert_train_dataset),
    ("Formal dev", bert_formal_dev_dataset),
    ("Informal dev", bert_informal_dev_dataset)
]:
    supervised_words = sum(
        sum(label != IGNORE_INDEX for label in ex["labels"])
        for ex in dataset.examples
    )

    print(
        f"{name:15s} | "
        f"{len(dataset):6,d} sentences | "
        f"{supervised_words:7,d} labeled words"
    )

print("\n=== PIPELINE STATUS ===")
print("POS categories:", len(pos2id))
print("Maximum token length:", MAX_LENGTH)
print("Dynamic padding: ENABLED")
print("Subword label alignment: VERIFIED")
print("Training: NOT STARTED")
print("Test datasets: NOT USED")

print("\nPARSBERT DATA PIPELINE READY")


In [ ]:

# ============================================
# FARSI SYNTAX LAB - EXPERIMENT 02
# CELL 10: EXPORT PARSBERT TRAINING DATA
# ============================================

import json
import zipfile
from pathlib import Path

# --------------------------------------------
# 1. Output location
# --------------------------------------------

output_dir = ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

bundle_path = output_dir / "parsbert_colab_bundle.zip"

datasets_to_export = {
    "formal_train.jsonl": bert_train_dataset,
    "formal_dev.jsonl": bert_formal_dev_dataset,
    "informal_dev.jsonl": bert_informal_dev_dataset
}

expected_words = {
    "formal_train.jsonl": 452496,
    "formal_dev.jsonl": 25147,
    "informal_dev.jsonl": 10049
}

# --------------------------------------------
# 2. Save metadata
# --------------------------------------------

metadata = {
    "project": "FarsiSyntax Lab",
    "experiment": "ParsBERT Persian POS Tagging",
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "ignore_index": IGNORE_INDEX,
    "seed": SEED,
    "pos2id": pos2id,
    "datasets": {}
}

# --------------------------------------------
# 3. Export tokenized datasets
# --------------------------------------------

print("=== EXPORTING PARSBERT DATASETS ===")

with zipfile.ZipFile(
    bundle_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for filename, dataset in datasets_to_export.items():

        word_count = 0

        with archive.open(filename, "w") as output:

            for example in dataset.examples:

                assert (
                    len(example["input_ids"])
                    == len(example["attention_mask"])
                    == len(example["labels"])
                )

                word_count += sum(
                    label != IGNORE_INDEX
                    for label in example["labels"]
                )

                line = json.dumps(
                    example,
                    ensure_ascii=False
                ) + "\n"

                output.write(line.encode("utf-8"))

        assert word_count == expected_words[filename], (
            f"Unexpected label count: {filename}"
        )

        metadata["datasets"][filename] = {
            "sentences": len(dataset),
            "labeled_words": word_count
        }

        print(
            f"[OK] {filename}: "
            f"{len(dataset):,} sentences, "
            f"{word_count:,} labels"
        )

    archive.writestr(
        "metadata.json",
        json.dumps(
            metadata,
            ensure_ascii=False,
            indent=2
        )
    )

# --------------------------------------------
# 4. Verify ZIP integrity
# --------------------------------------------

with zipfile.ZipFile(bundle_path, "r") as archive:

    assert archive.testzip() is None

    print("\n=== ARCHIVE CONTENTS ===")

    for item in archive.infolist():
        print(
            f"{item.filename}: "
            f"{item.file_size / 1024:.1f} KB"
        )

print("\n=== GPU TRAINING BUNDLE READY ===")
print("Location:", bundle_path)
print(
    "Compressed size:",
    f"{bundle_path.stat().st_size / (1024**2):.2f} MB"
)
print("POS categories:", len(pos2id))
print("Test datasets included: NO")
print("Training started: NO")
